# Ablation kiến trúc MNN: từ MNN gốc (x, y tuyệt đối) tới mô hình đề xuất

Notebook này điền bảng **Ablation Study đề xuất** (RMSE +6h và +12h) trên `ibtracs_tbd_6h.csv`, cùng split, cùng chuẩn hoá và cùng quy trình huấn luyện cho mọi dòng:

| Dòng | Cấu hình | Đầu vào | Mục tiêu | Lớp | LayerNorm | Kích hoạt |
|---|---|---|---|---|---|---|
| R1 | Original MNN | **lon, lat tuyệt đối** | vị trí tuyệt đối tại t+h | song tuyến tính dùng chung (U·X·Vᵀ) | không | sigmoid |
| R2 | MNN + displacement | dx, dy (km) | độ dời km | song tuyến tính | không | sigmoid |
| R3 | MNN + speed/direction | dx, dy, speed, sin/cos hướng | độ dời km | song tuyến tính | không | sigmoid |
| R4 | M-neuron thay bilinear | như R3 | độ dời km | **M-neuron** | không | sigmoid |
| R5 | M-neuron + LayerNorm | như R3 | độ dời km | M-neuron | **có** | sigmoid |
| R6 | M-neuron + algebraic activation | như R3 | độ dời km | M-neuron | không | **algebraic** |
| R7 | Full proposed model | như R3 | độ dời km | M-neuron | có | algebraic |
| R7-leak | Full, speed/dir IBTrACS gốc | như R3 nhưng speed/dir tại t | độ dời km | M-neuron | có | algebraic |

- Speed/direction ở R3–R7 là bản **nhân quả** (`SPEED_SRC = 'lag2'`: STORM_SPEED/DIR lùi 2 bước; hoặc `'raw'`: tự tính sai phân lùi).
- **R7-leak** dùng STORM_SPEED/DIR gốc tại t (bị rò rỉ), để kiểm chứng hai số đã báo cáo 15,332 và 63,83 km. Không dùng dòng này làm kết quả.
- Lớp song tuyến tính dùng chung: Y = σ(U·X·Vᵀ + B), như Gao (2017) / Zhang (2018). M-neuron: mỗi phần tử đầu ra có cặp trọng số W_L, W_R riêng.
- L2 = 0,0005 cho mọi dòng (trên U, V hoặc W_L, W_R).

Kết quả từng lần chạy lưu trong `KetQua_KienTruc/`; chạy lại sẽ bỏ qua những lần đã có.

## Thư viện

Notebook dùng `numpy`, `pandas`, `matplotlib`, `scikit-learn` và `tensorflow` (bản 2.x, gọi qua `tf.compat.v1`).
Chạy ô dưới để xem phiên bản đang dùng:

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'   # tránh crash kernel do trùng thư viện OpenMP trên Windows

import time
import random
import shutil

import numpy as np
import tensorflow as tf2              # import TF trước matplotlib/sklearn
import tensorflow.compat.v1 as tf
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

tf.disable_v2_behavior()
tf2.get_logger().setLevel('ERROR')

plt.rc('font', **{'size': 12})
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 120)

for name, mod in [('numpy', np), ('pandas', pd), ('matplotlib', matplotlib),
                  ('sklearn', sklearn), ('tensorflow', tf2)]:
    print(f'{name:11s} {mod.__version__}')

## 0. Cấu hình chung

`QUICK = True` chạy nhanh 4 epoch, 1 seed để kiểm tra đường ống.

In [ ]:
SEED = 42
CSV_PATH = 'ibtracs_tbd_6h.csv'
for _p in [CSV_PATH, os.path.join('..', CSV_PATH)]:
    if os.path.exists(_p):
        CSV_PATH = _p
        break
print('Dùng dữ liệu:', os.path.abspath(CSV_PATH))
OUT_DIR = 'KetQua_KienTruc'

FORECAST_LIST = [1, 2]          # +6h và +12h
WINDOW_SIZE = 4
FIRST_HID = [2, 25, 8]
SECOND_HID = [2, 50, 10]
TWO_STAGE = False
MAX_EPOCHS = 200
STOP_PATIENCE = 30
L2_LAMBDA = 5e-4
SEEDS = [42, 43, 44]
N_BOOT = 2000
SPEED_SRC = 'lag2'              # 'lag2' (STORM_SPEED/DIR lùi 2 bước) hoặc 'raw' (tự tính sai phân lùi)
BASE_ACT = 'sigmoid'            # kích hoạt của các dòng không dùng algebraic

MAX_STEP_KM = 600.0
MIN_SEG_LEN = WINDOW_SIZE + max(FORECAST_LIST) + 1
NATURE_KEEP = None
KM_PER_DEG = 111.195
COL_LAT_ABS = 1                 # bảng tuyệt đối: cột 0 = kinh độ (0..360, liên tục), cột 1 = vĩ độ

QUICK = False
if QUICK:
    MAX_EPOCHS, SEEDS = 4, [42]

random.seed(SEED)
np.random.seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Dữ liệu

Đọc, cắt phân đoạn và chia tập **giống hệt** `MNN_TongHop.ipynb` (cùng `load_csv_data`, `split_by_storm`), nên tập test trùng với các thí nghiệm trước.

In [ ]:
def haversine_distance(coords_true, coords_pred):
    """Khoảng cách Haversine (km); mỗi hàng là (lon, lat) theo độ."""
    R = 6371.0
    lon1, lat1 = np.radians(coords_true[:, 0]), np.radians(coords_true[:, 1])
    lon2, lat2 = np.radians(coords_pred[:, 0]), np.radians(coords_pred[:, 1])
    a = (np.sin((lat2 - lat1) / 2.0) ** 2 +
         np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2.0) ** 2)
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def load_csv_data(csv_path, step_hours=6):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Không tìm thấy file '{csv_path}'!")

    df = pd.read_csv(csv_path, encoding='utf-8', low_memory=False, keep_default_na=False, na_values=[''])
    df['BASIN'] = df['BASIN'].fillna('NA')
    df['ISO_TIME'] = pd.to_datetime(df['ISO_TIME'], errors='coerce')
    for c in ['STORM_SPEED', 'STORM_DIR', 'LAT', 'LON']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df[['STORM_SPEED', 'STORM_DIR']] = df.groupby('ID', sort=False)[['STORM_SPEED', 'STORM_DIR']].ffill().fillna(0.0)
    df = df.dropna(subset=['ISO_TIME', 'LAT', 'LON'])
    if NATURE_KEEP is not None:
        df = df[df['NATURE'].isin(NATURE_KEEP)]

    cyclones_data, initial_coords, meta = [], [], []
    n_jump_cuts = 0

    for storm_id, group in df.groupby('ID', sort=False):
        group = group.sort_values('ISO_TIME')
        lon = group['LON'].to_numpy(np.float64)
        lat = group['LAT'].to_numpy(np.float64)

        dt_h = group['ISO_TIME'].diff().dt.total_seconds().to_numpy() / 3600.0
        step_km = np.r_[0.0, haversine_distance(np.c_[lon[:-1], lat[:-1]], np.c_[lon[1:], lat[1:]])]
        gap = dt_h[1:] != step_hours
        jump = step_km[1:] > MAX_STEP_KM
        n_jump_cuts += int(np.sum(jump & ~gap))
        breaks = np.where(gap | jump)[0] + 1

        for seg_no, idx in enumerate(np.split(np.arange(len(group)), breaks)):
            if len(idx) < MIN_SEG_LEN:
                continue
            seg = group.iloc[idx]
            s_lon, s_lat = lon[idx], lat[idx]
            dir_rad = np.radians(seg['STORM_DIR'].to_numpy(np.float64))

            start_lon, start_lat = s_lon[0], s_lat[0]
            rel_lon = (s_lon - start_lon + 180.0) % 360.0 - 180.0
            rel_lat = s_lat - start_lat

            cols = [rel_lon, rel_lat, seg['STORM_SPEED'].to_numpy(np.float64),
                    np.sin(dir_rad), np.cos(dir_rad), s_lat]
            cyclones_data.append(np.column_stack(cols))
            initial_coords.append((start_lon, start_lat))

            name, year = str(seg['NAME'].iloc[0]).strip(), int(seg['YEAR'].iloc[0])
            meta.append({'id': storm_id, 'name': name, 'year': year, 'basin': str(seg['BASIN'].iloc[0]),
                         'nature': seg['NATURE'].astype(str).to_numpy(), 'seg': seg_no,
                         'label': f'{name}_{year}_ID{storm_id}_s{seg_no}'})
    print(f'Số chỗ cắt do nhảy > {MAX_STEP_KM:.0f} km: {n_jump_cuts}')
    return cyclones_data, initial_coords, meta

In [ ]:
all_cyc, all_init, meta = load_csv_data(CSV_PATH)
print(f'{len(all_cyc)} phân đoạn từ {len({m["id"] for m in meta})} cơn bão, '
      f'{sum(t.shape[0] for t in all_cyc)} điểm quỹ đạo')
pd.DataFrame(all_cyc[0], columns=['dlon', 'dlat', 'speed_kt', 'sin_dir', 'cos_dir', 'lat_abs']).head()

In [ ]:
def split_by_storm(meta, f_train=0.7, f_val=0.85):
    ids = list(dict.fromkeys(m['id'] for m in meta))
    a, b = int(len(ids) * f_train), int(len(ids) * f_val)
    sets = [set(ids[:a]), set(ids[a:b]), set(ids[b:])]
    return [[i for i, m in enumerate(meta) if m['id'] in s] for s in sets]


split = split_by_storm(meta, 0.7, 0.85)
names = ['Huấn luyện', 'Kiểm định', 'Kiểm thử']

rows, id_sets = [], []
for name, idx in zip(names, split):
    ids = {meta[i]['id'] for i in idx}
    id_sets.append(ids)
    years = [meta[i]['year'] for i in idx]
    rows.append({'Tập': name, 'Số cơn bão': len(ids), 'Số phân đoạn': len(idx),
                 'Số điểm quỹ đạo': sum(all_cyc[i].shape[0] for i in idx),
                 **{f'Số cửa sổ +{6 * fs}h': sum(max(all_cyc[i].shape[0] - WINDOW_SIZE - fs, 0) for i in idx)
                    for fs in FORECAST_LIST},
                 'Năm': f'{min(years)}-{max(years)}'})
split_stats = pd.DataFrame(rows)
split_stats['Tỉ lệ cơn (%)'] = (100 * split_stats['Số cơn bão'] / split_stats['Số cơn bão'].sum()).round(2)
display(split_stats)

for a in range(3):
    for b in range(a + 1, 3):
        print(f'Trùng ID {names[a]} ∩ {names[b]}: {len(id_sets[a] & id_sets[b])} cơn')

### 1.1 Bảng vị trí tuyệt đối và các nguồn speed/direction

Đổi mỗi phân đoạn sang bảng tuyệt đối: cột 0 = kinh độ (đưa điểm đầu về 0..360 để các cơn ở Thái Bình Dương không nhảy qua 180°), cột 1 = vĩ độ, cột 2–4 = speed, sin, cos hướng.

- `ib`: STORM_SPEED/DIR gốc của IBTrACS (rò rỉ).
- `lag2`: lùi 2 bước (nhân quả).
- `raw`: tự tính sai phân lùi từ vị trí (nhân quả).

In [ ]:
def step_km(lon, lat):
    """Độ dời km (Đông, Bắc) giữa các điểm liền kề, dùng vĩ độ trung điểm."""
    latm = np.radians((lat[1:] + lat[:-1]) / 2.0)
    dlon = (np.diff(lon) + 180.0) % 360.0 - 180.0
    return dlon * KM_PER_DEG * np.cos(latm), np.diff(lat) * KM_PER_DEG


def raw_velocity(lon, lat, dt=6.0):
    """Sai phân lùi (km/h). Hàng 0 chép từ hàng 1 (hàng 0 không bao giờ nằm trong cửa sổ)."""
    dx, dy = step_km(lon, lat)
    V = np.c_[dx, dy] / dt
    return np.vstack([V[:1], V])


def kalman_velocity(lon, lat, dt=6.0, sigma_pos=15.0, sigma_acc=0.5):
    """Kalman nhân quả; sigma_pos: sai số vị trí (km), sigma_acc: nhiễu gia tốc (km/h^2)."""
    dx, dy = step_km(lon, lat)
    pos = np.c_[np.r_[0.0, np.cumsum(dx)], np.r_[0.0, np.cumsum(dy)]]
    n = len(pos)
    A = np.array([[1, 0, dt, 0], [0, 1, 0, dt], [0, 0, 1, 0], [0, 0, 0, 1]], float)
    Q1 = sigma_acc ** 2 * np.array([[dt ** 3 / 3, dt ** 2 / 2], [dt ** 2 / 2, dt]])
    Q = np.zeros((4, 4))
    Q[np.ix_([0, 2], [0, 2])] = Q1
    Q[np.ix_([1, 3], [1, 3])] = Q1
    H = np.array([[1, 0, 0, 0], [0, 1, 0, 0]], float)
    R = np.eye(2) * sigma_pos ** 2
    x = np.array([pos[0, 0], pos[0, 1], 0.0, 0.0])
    P = np.diag([sigma_pos ** 2, sigma_pos ** 2, 30.0 ** 2, 30.0 ** 2])
    V = np.zeros((n, 2))
    for k in range(1, n):
        x = A @ x
        P = A @ P @ A.T + Q
        K = P @ H.T @ np.linalg.inv(H @ P @ H.T + R)
        x = x + K @ (pos[k] - H @ x)
        P = (np.eye(4) - K @ H) @ P
        V[k] = x[2:]
    V[0] = V[1]
    return V


def speed_sin_cos(V, eps=1e-6):
    """(tốc độ km/h, sin(hướng), cos(hướng)); hướng đo từ Bắc theo chiều kim đồng hồ."""
    sp = np.hypot(V[:, 0], V[:, 1])
    return sp, V[:, 0] / (sp + eps), V[:, 1] / (sp + eps)


def with_vel(t, mode, sigma_pos=15.0, sigma_acc=0.5):
    """Thay cột 2, 3, 4 bằng speed (knot), sin, cos tự tính từ tọa độ."""
    lon, lat = t[:, 0], t[:, COL_LAT_ABS]
    V = raw_velocity(lon, lat) if mode == 'raw' else kalman_velocity(lon, lat, sigma_pos=sigma_pos, sigma_acc=sigma_acc)
    sp, s, c = speed_sin_cos(V)
    t = t.copy()
    t[:, 2], t[:, 3], t[:, 4] = sp / 1.852, s, c
    return t


def lag_ib(t, k=2):
    """Lùi STORM_SPEED/DIR k bước. Phần 7.1: giá trị IBTrACS tại hàng r còn chứa thông tin tới vị trí r+2,
    nên phải lùi k = 2 mới hết rò rỉ (k = 1 vẫn rò rỉ)."""
    t = t.copy()
    t[k:, 2:5] = t[:-k, 2:5]
    return t

In [ ]:
def abs_table(t, init):
    a = t.copy()
    a[:, 0] = t[:, 0] + (init[0] % 360.0)
    a[:, 1] = t[:, 1] + init[1]
    return a


abs_cyc = [abs_table(t, i) for t, i in zip(all_cyc, all_init)]
SRC = {'ib': abs_cyc,
       'lag2': [lag_ib(t, 2) for t in abs_cyc],
       'raw': [with_vel(t, 'raw') for t in abs_cyc]}
zero_init = [(0.0, 0.0)] * len(abs_cyc)
tr_idx, va_idx, te_idx = split
print('Ví dụ bảng tuyệt đối (lon, lat, speed, sin, cos, lat):')
print(abs_cyc[0][:3].round(3))

## 2. Tạo cửa sổ đầu vào

Tên đặc trưng: `lon_abs`, `lat_abs` (vị trí tuyệt đối, độ), `dx`, `dy` (độ dời km từng bước 6h), `spd`, `sin`, `cos` (cột 2–4 của bảng).

**Khi đầu vào có `lon_abs`/`lat_abs` thì mục tiêu là vị trí tuyệt đối tại t+h** (như MNN gốc); các trường hợp khác mục tiêu là độ dời km từ điểm neo.

In [ ]:
NAME2COL = {'lon_abs': 0, 'lat_abs': 1, 'spd': 2, 'sin': 3, 'cos': 4}


def deg_to_km(dlon, dlat, lat_ref):
    coslat = np.cos(np.radians(lat_ref))
    return dlon * KM_PER_DEG * coslat, dlat * KM_PER_DEG


def km_to_deg(dx, dy, lat_ref):
    coslat = np.cos(np.radians(lat_ref))
    return dx / (KM_PER_DEG * coslat), dy / KM_PER_DEG


def anchor_indices(n_rows, window_size, forecast_steps=1):
    return np.arange(window_size, n_rows - forecast_steps)


def is_abs_target(feature_cols):
    return 'lon_abs' in feature_cols or 'lat_abs' in feature_cols


def data_processing(table, window_size, forecast_steps=1, feature_cols=('dx', 'dy')):
    js = anchor_indices(table.shape[0], window_size, forecast_steps)
    nf = len(feature_cols)
    if len(js) == 0:
        return np.zeros((0, window_size, nf)), np.zeros((0, 1, 2))
    absolute = is_abs_target(feature_cols)
    X = np.empty((len(js), window_size, nf))
    Y = np.empty((len(js), 1, 2))
    for n, j in enumerate(js):
        lat_ref = table[j, COL_LAT_ABS]
        steps = np.diff(table[j - window_size: j + 1, :2], axis=0)
        dx, dy = deg_to_km(steps[:, 0], steps[:, 1], lat_ref)
        for q, c in enumerate(feature_cols):
            X[n, :, q] = dx if c == 'dx' else dy if c == 'dy' else table[j - window_size + 1: j + 1, NAME2COL[c]]
        if absolute:
            Y[n, 0] = table[j + forecast_steps, :2]
        else:
            tgt = table[j + forecast_steps, :2] - table[j, :2]
            Y[n, 0] = deg_to_km(tgt[0], tgt[1], lat_ref)
    return X, Y


def persistence_km(table, window_size, forecast_steps=1):
    js = anchor_indices(table.shape[0], window_size, forecast_steps)
    last = table[js, :2] - table[js - 1, :2]
    dx, dy = deg_to_km(last[:, 0], last[:, 1], table[js, COL_LAT_ABS])
    return forecast_steps * np.c_[dx, dy]

## 3. Các biến thể kiến trúc

`arch = {'layer': 'bilinear' | 'mneuron', 'ln': True/False, 'act': 'sigmoid' | 'tanh' | 'algebraic'}`.

- **bilinear** (dùng chung): u = U·X·Vᵀ + B, với U ∈ ℝ^{m1×n1}, V ∈ ℝ^{m2×n2}.
- **mneuron**: u[j,k] = Σ_pq X[p,q]·W_L[j,k,p]·W_R[j,k,q] + θ[j,k] (mô hình hiện tại).

Hai lớp ẩn dùng `act` và (tuỳ chọn) LayerNorm; lớp ra cùng loại lớp, không LayerNorm, không kích hoạt.

In [ ]:
def algebraic_activation(x):
    return x / tf.sqrt(1.0 + tf.square(x))


ACTS = {'sigmoid': tf.sigmoid, 'tanh': tf.tanh, 'algebraic': algebraic_activation, None: None}


def matrix_layer(inputs, in_shape, out_shape, layer_type='mneuron', activation=None, use_layernorm=False,
                 layer_name='m_layer', drop_rate=None):
    n1, n2 = in_shape[1], in_shape[2]
    m1, m2 = out_shape[1], out_shape[2]
    with tf.variable_scope(layer_name):
        if layer_type == 'bilinear':
            U = tf.Variable(tf.random.normal([m1, n1], stddev=n1 ** -0.5), name='W_L')
            V = tf.Variable(tf.random.normal([m2, n2], stddev=n2 ** -0.5), name='W_R')
            u = tf.einsum('jp,bpq,kq->bjk', U, inputs, V)
        else:
            s = (n1 * n2) ** -0.25
            W = tf.Variable(tf.random.normal([m1, m2, n1], stddev=s), name='W_L')
            V = tf.Variable(tf.random.normal([m1, m2, n2], stddev=s), name='W_R')
            u = tf.einsum('bpq,jkp,jkq->bjk', inputs, W, V)
        u = u + tf.Variable(tf.zeros([1, m1, m2]), name='theta')
        if use_layernorm:
            mean, variance = tf.nn.moments(u, axes=[1, 2], keepdims=True)
            gamma = tf.Variable(tf.ones([1, m1, m2]), name='gamma')
            beta = tf.Variable(tf.zeros([1, m1, m2]), name='beta')
            u = gamma * ((u - mean) / tf.sqrt(variance + 1e-6)) + beta
        out = u if activation is None else activation(u)
        if drop_rate is not None:
            out = tf.nn.dropout(out, rate=drop_rate)
    return out


def build_mnn(xs, window_size, n_feat, first_hid, second_hid, drop_rate=None, arch=None):
    arch = arch or {'layer': 'mneuron', 'ln': True, 'act': 'algebraic'}
    act, lt, ln = ACTS[arch['act']], arch['layer'], arch['ln']
    l1 = matrix_layer(xs, [None, window_size, n_feat], first_hid, lt, act, ln, 'layer1', drop_rate)
    l2 = matrix_layer(l1, first_hid, second_hid, lt, act, ln, 'layer2', drop_rate)
    return matrix_layer(l2, second_hid, [None, 1, 2], lt, None, False, 'layer3')

## 4. Huấn luyện

Giống `Training.py`: Adam, lr 1e-3, batch 256, clip gradient ±1, dừng sớm khi `stop_patience` epoch liền val không giảm quá 0,2%, lưu checkpoint tại val tốt nhất.

Hai thay đổi:
- Thư mục checkpoint mặc định nằm trong thư mục làm việc (notebook không có `__file__`).
- `lr_patience` trong bản gốc khai báo nhưng không dùng (lr luôn cố định). Ở đây để mặc định `None` (giữ nguyên hành vi cũ); đặt số nguyên để bật giảm lr ×0,5 khi val chững lại.

In [ ]:
def _stack(data, window_size, forecast_steps, feature_cols):
    Xs, Ys = [], []
    for table in data:
        X, Y = data_processing(table, window_size, forecast_steps, feature_cols)
        if len(X) > 0:
            Xs.append(X)
            Ys.append(Y)
    return np.concatenate(Xs, axis=0), np.concatenate(Ys, axis=0)


def training(data_train, data_val, window_size, first_hid, second_hid, seed=None, forecast_steps=1,
             feature_cols=(0, 1, 2, 3, 4), arch=None, save_dir=None, two_stage=False, batch_size=256,
             max_epochs=200, learning_rate=0.001, lr_patience=None, stop_patience=30,
             min_rel_improve=2e-3, min_lr=1e-5, verbose=True, drop_rate=0.0, l2_lambda=0.0):
    tf.reset_default_graph()
    seed = 0 if seed is None else seed
    tf.set_random_seed(seed)
    rng = np.random.RandomState(seed)

    X_tr, Y_tr = _stack(data_train, window_size, forecast_steps, feature_cols)
    X_va, Y_va = _stack(data_val, window_size, forecast_steps, feature_cols)
    n_feat = X_tr.shape[2]

    scaler_X, scaler_Y = StandardScaler(), StandardScaler()
    scaler_X.fit(X_tr.reshape(-1, n_feat))
    scaler_Y.fit(Y_tr.reshape(-1, 2))
    sx = lambda X: scaler_X.transform(X.reshape(-1, n_feat)).reshape(-1, window_size, n_feat).astype(np.float32)
    sy = lambda Y: scaler_Y.transform(Y.reshape(-1, 2)).reshape(-1, 1, 2).astype(np.float32)
    X_tr_s, Y_tr_s, X_va_s, Y_va_s = sx(X_tr), sy(Y_tr), sx(X_va), sy(Y_va)
    if verbose:
        print(f'   Mẫu train: {len(X_tr_s)} | mẫu val: {len(X_va_s)} | số đặc trưng: {n_feat}')

    xs = tf.placeholder(tf.float32, [None, window_size, n_feat])
    ys = tf.placeholder(tf.float32, [None, 1, 2])
    lr_ph = tf.placeholder(tf.float32, [])
    drop_ph = tf.placeholder_with_default(0.0, [], name='drop_rate') if drop_rate > 0 else None
    prediction = build_mnn(xs, window_size, n_feat, first_hid, second_hid, drop_rate=drop_ph, arch=arch)
    loss = tf.reduce_mean(tf.square(ys - prediction))
    if l2_lambda > 0:
        l2 = tf.add_n([tf.nn.l2_loss(v) for v in tf.trainable_variables() if 'W_L' in v.name or 'W_R' in v.name])
        train_loss = loss + l2_lambda * l2
    else:
        train_loss = loss

    def make_train_op(var_list):
        opt = tf.train.AdamOptimizer(learning_rate=lr_ph)
        gv = opt.compute_gradients(train_loss, var_list)
        gv = [(tf.clip_by_value(g, -1.0, 1.0), v) for g, v in gv if g is not None]
        return opt.apply_gradients(gv)

    all_vars = tf.trainable_variables()
    if two_stage:
        train_ops = [make_train_op([v for v in all_vars if 'W_R' not in v.name]),
                     make_train_op([v for v in all_vars if 'W_R' in v.name])]
    else:
        train_ops = [make_train_op(all_vars)]

    if save_dir is None:
        save_dir = os.path.join(os.getcwd(), f'saved_model_{forecast_steps}')
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, 'model')

    sess = tf.Session()
    sess.run(tf.global_variables_initializer())
    saver = tf.train.Saver(tf.trainable_variables())

    def full_loss(X, Y, chunk=8192):
        tot = 0.0
        for i in range(0, len(X), chunk):
            tot += sess.run(loss, {xs: X[i:i + chunk], ys: Y[i:i + chunk]}) * len(X[i:i + chunk])
        return tot / len(X)

    train_losses, val_losses = [], []
    best_val, best_epoch, lr = np.inf, -1, learning_rate
    ref_val, since_best, since_lr = np.inf, 0, 0
    n = len(X_tr_s)

    for epoch in range(max_epochs):
        perm = rng.permutation(n)
        for i in range(0, n, batch_size):
            b = perm[i:i + batch_size]
            fd = {xs: X_tr_s[b], ys: Y_tr_s[b], lr_ph: lr}
            if drop_ph is not None:
                fd[drop_ph] = drop_rate
            for op in train_ops:
                sess.run(op, fd)

        tr_l, va_l = full_loss(X_tr_s, Y_tr_s), full_loss(X_va_s, Y_va_s)
        train_losses.append(tr_l)
        val_losses.append(va_l)

        if va_l < best_val:
            best_val, best_epoch = va_l, epoch
            saver.save(sess, save_path)

        if va_l < ref_val * (1.0 - min_rel_improve):
            ref_val, since_best, since_lr = va_l, 0, 0
        else:
            since_best += 1
            since_lr += 1
            if lr_patience is not None and since_lr >= lr_patience and lr > min_lr:
                lr, since_lr = max(lr * 0.5, min_lr), 0
                if verbose:
                    print(f'   epoch {epoch}: giảm lr xuống {lr:.1e}')
            if since_best >= stop_patience:
                if verbose:
                    print(f'   Dừng sớm ở epoch {epoch}: {stop_patience} epoch liền val không giảm quá '
                          f'{100 * min_rel_improve:.1f}%')
                break

        if verbose and epoch % 10 == 0:
            print(f'   epoch {epoch:3d}  train {tr_l:.5f}  val {va_l:.5f}  lr {lr:.1e}')

    sess.close()
    if verbose:
        print(f'   Best val loss {best_val:.5f} tại epoch {best_epoch}')
    return scaler_X, scaler_Y, train_losses, val_losses, best_epoch

## 5. Kiểm thử

Khôi phục checkpoint tốt nhất, dự báo từng phân đoạn và đổi ngược về km.

In [ ]:
def testing(data_test, window_size, first_hid, second_hid, scaler_X, scaler_Y, forecast_steps=1,
            feature_cols=(0, 1, 2, 3, 4), save_dir=None, arch=None):
    """Trả về (Y_true_km, Y_pred_km, valid_indices)."""
    tf.reset_default_graph()
    n_feat = scaler_X.n_features_in_
    xs = tf.placeholder(tf.float32, [None, window_size, n_feat])
    prediction = build_mnn(xs, window_size, n_feat, first_hid, second_hid, arch=arch)

    if save_dir is None:
        save_dir = os.path.join(os.getcwd(), f'saved_model_{forecast_steps}')
    sess = tf.Session()
    tf.train.Saver(tf.trainable_variables()).restore(sess, os.path.join(save_dir, 'model'))

    Y_true, Y_pred, valid = [], [], []
    for idx, storm in enumerate(data_test):
        X, Y = data_processing(storm, window_size, forecast_steps, feature_cols)
        if len(X) == 0:
            continue
        valid.append(idx)
        X_s = scaler_X.transform(X.reshape(-1, n_feat)).reshape(-1, window_size, n_feat)
        p = sess.run(prediction, feed_dict={xs: X_s})
        Y_true.append(Y.reshape(-1, 2))
        Y_pred.append(scaler_Y.inverse_transform(p.reshape(-1, 2)))
    sess.close()
    return Y_true, Y_pred, valid


def to_abs(table, initial, disp_km, forecast_steps):
    """Độ dời km từ điểm neo -> toạ độ tuyệt đối (lon, lat)."""
    js = anchor_indices(table.shape[0], WINDOW_SIZE, forecast_steps)
    anchor = table[js, :2] + np.array(initial)
    dlon, dlat = km_to_deg(disp_km[:, 0], disp_km[:, 1], table[js, COL_LAT_ABS])
    return anchor + np.c_[dlon, dlat]


def hav_errors(tables, initials, valid, true_km, pred_km, fs):
    """Sai số Haversine (km) của mọi điểm dự báo, nối theo thứ tự phân đoạn."""
    return np.concatenate([haversine_distance(to_abs(tables[i], initials[i], t, fs),
                                              to_abs(tables[i], initials[i], p, fs))
                           for i, t, p in zip(valid, true_km, pred_km)])

## 6. Chạy ablation kiến trúc

In [ ]:
SPD_FEATS = ['dx', 'dy', 'spd', 'sin', 'cos']
ARCHS = {
    'R1':      ('Original MNN, x/y tuyệt đối',          ['lon_abs', 'lat_abs'], 'ib',      'bilinear', False, BASE_ACT,    True),
    'R2':      ('MNN + displacement',                    ['dx', 'dy'],           'ib',      'bilinear', False, BASE_ACT,    True),
    'R3':      ('MNN + speed/direction',                 SPD_FEATS,              SPEED_SRC, 'bilinear', False, BASE_ACT,    True),
    'R4':      ('M-neuron thay bilinear',                SPD_FEATS,              SPEED_SRC, 'mneuron',  False, BASE_ACT,    True),
    'R5':      ('M-neuron + LayerNorm',                  SPD_FEATS,              SPEED_SRC, 'mneuron',  True,  BASE_ACT,    True),
    'R6':      ('M-neuron + algebraic activation',       SPD_FEATS,              SPEED_SRC, 'mneuron',  False, 'algebraic', True),
    'R7':      ('Full proposed model',                   SPD_FEATS,              SPEED_SRC, 'mneuron',  True,  'algebraic', True),
    'R7-leak': ('Full, speed/dir IBTrACS gốc (rò rỉ)',   SPD_FEATS,              'ib',      'mneuron',  True,  'algebraic', False),
}
RUN = list(ARCHS)                    # bỏ 'R7-leak' nếu không cần kiểm chứng số cũ
COMPARISONS = [('R2', 'R1'), ('R3', 'R2'), ('R4', 'R3'), ('R5', 'R4'), ('R6', 'R4'), ('R7', 'R4'), ('R7', 'R1'), ('R7-leak', 'R7')]


def run_file(key, fs, seed):
    lab, feats, src, layer, ln, act, _ = ARCHS[key]
    tag = f'{key}_{src}_{layer}_ln{int(ln)}_{act}'
    return os.path.join(OUT_DIR, f'{tag}_fs{fs}_l2{L2_LAMBDA:g}_ep{MAX_EPOCHS}_s{seed}.npz')


def model_errors(tabs, valid, yt, yp, fs, feats):
    if is_abs_target(feats):
        return np.concatenate([haversine_distance(t, p) for t, p in zip(yt, yp)])
    return hav_errors(tabs, zero_init, valid, yt, yp, fs)


def run_one(key, fs, seed):
    path = run_file(key, fs, seed)
    if os.path.exists(path):
        return False
    lab, feats, src, layer, ln, act, _ = ARCHS[key]
    arch = {'layer': layer, 'ln': ln, 'act': act}
    T = SRC[src]
    tr, va, te = ([T[i] for i in idx] for idx in split)
    sd = os.path.join(OUT_DIR, 'ckpt', f'{key}_{fs}_{seed}')
    t0 = time.time()
    sX, sY, trl, vll, best = training(tr, va, WINDOW_SIZE, FIRST_HID, SECOND_HID, seed=seed, forecast_steps=fs,
                                      feature_cols=feats, arch=arch, save_dir=sd, two_stage=TWO_STAGE,
                                      max_epochs=MAX_EPOCHS, stop_patience=STOP_PATIENCE, l2_lambda=L2_LAMBDA,
                                      verbose=False)
    out = {'best_epoch': best}
    for name, tabs in (('val', va), ('test', te)):
        yt, yp, valid = testing(tabs, WINDOW_SIZE, FIRST_HID, SECOND_HID, sX, sY, fs, feats, save_dir=sd, arch=arch)
        out[f'err_{name}'] = model_errors(tabs, valid, yt, yp, fs, feats)
    np.savez(path, **out)
    shutil.rmtree(sd, ignore_errors=True)
    et = out['err_test']
    print(f'test Mean {et.mean():6.2f} RMSE {np.sqrt(np.mean(et ** 2)):6.2f} km | best_epoch {best} | {time.time() - t0:.0f}s')
    return True


jobs = [(k, fs, s) for fs in FORECAST_LIST for s in SEEDS for k in RUN]
print(f'{len(jobs)} lần chạy')
for n, (k, fs, s) in enumerate(jobs, 1):
    print(f'[{n}/{len(jobs)}] {k} +{6 * fs}h seed={s}: ', end='')
    if not run_one(k, fs, s):
        print('đã có, bỏ qua')

## 7. Bảng kết quả

- **Bảng gọn** đúng mẫu báo cáo: RMSE +6h và +12h (trung bình 3 seed ± độ lệch chuẩn).
- **Bảng đầy đủ** cho từng hạn: Mean, RMSE, Median, P95, skill so với persistence.
- **So sánh cặp**: Δ RMSE và Δ Mean trên test, CI95 bootstrap theo cơn bão (Δ < 0: dòng bên trái tốt hơn).

In [ ]:
def rmse(e):
    return float(np.sqrt(np.mean(e ** 2)))


def boot_delta(Ex, Ey, groups, n_boot=N_BOOT, seed=SEED):
    G = groups.max() + 1
    cnt = np.bincount(groups, minlength=G).astype(float)
    S1 = lambda E: np.stack([np.bincount(groups, e, G) for e in E])
    S2 = lambda E: np.stack([np.bincount(groups, e ** 2, G) for e in E])
    s1x, s2x, s1y, s2y = S1(Ex), S2(Ex), S1(Ey), S2(Ey)
    w = np.random.default_rng(seed).multinomial(G, np.full(G, 1.0 / G), size=n_boot).astype(float)
    nb = w @ cnt
    d_mean = ((w @ s1x.T) - (w @ s1y.T)).mean(axis=1) / nb
    d_rmse = (np.sqrt((w @ s2x.T) / nb[:, None]) - np.sqrt((w @ s2y.T) / nb[:, None])).mean(axis=1)
    pm = np.mean(Ex.mean(axis=1) - Ey.mean(axis=1))
    pr = np.mean(np.sqrt((Ex ** 2).mean(axis=1)) - np.sqrt((Ey ** 2).mean(axis=1)))
    return pm, np.percentile(d_mean, [2.5, 97.5]), pr, np.percentile(d_rmse, [2.5, 97.5])


te_tabs = [abs_cyc[i] for i in te_idx]
compact, full_tabs = {}, {}
for fs in FORECAST_LIST:
    groups = pd.factorize(np.concatenate([[meta[i]['id']] * len(anchor_indices(abs_cyc[i].shape[0], WINDOW_SIZE, fs))
                                          for i in te_idx]))[0]
    true_km = [data_processing(t, WINDOW_SIZE, fs, ('dx',))[1].reshape(-1, 2) for t in te_tabs]
    e_per = hav_errors(te_tabs, zero_init, range(len(te_tabs)), true_km,
                       [persistence_km(t, WINDOW_SIZE, fs) for t in te_tabs], fs)
    E, rows = {}, []
    for key in ARCHS:
        files = [run_file(key, fs, s) for s in SEEDS if os.path.exists(run_file(key, fs, s))]
        if not files:
            continue
        et = np.stack([np.load(f)['err_test'] for f in files])
        assert et.shape[1] == len(e_per)
        E[key] = et
        m, r = et.mean(axis=1), np.sqrt((et ** 2).mean(axis=1))
        sd = (lambda v: v.std(ddof=1)) if len(files) > 1 else (lambda v: np.nan)
        lab, feats, src, layer, ln, act, causal = ARCHS[key]
        rows.append({'Dòng': key, 'Configuration': lab, 'Nhân quả': causal, 'n_seed': len(files),
                     'Mean (km)': m.mean(), 'Std Mean': sd(m), 'RMSE (km)': r.mean(), 'Std RMSE': sd(r),
                     'Median (km)': np.median(et, axis=1).mean(), 'P95 (km)': np.percentile(et, 95, axis=1).mean()})
        compact.setdefault(key, {'Configuration': lab})[f'{6 * fs}h RMSE'] = (
            f'{r.mean():.3f}' + (f' ± {sd(r):.3f}' if len(files) > 1 else ''))
    rows.append({'Dòng': '—', 'Configuration': 'Persistence', 'Nhân quả': True, 'Mean (km)': e_per.mean(),
                 'RMSE (km)': rmse(e_per), 'Median (km)': np.median(e_per), 'P95 (km)': np.percentile(e_per, 95)})
    compact.setdefault('—', {'Configuration': 'Persistence'})[f'{6 * fs}h RMSE'] = f'{rmse(e_per):.3f}'
    tab = pd.DataFrame(rows)
    tab['Skill Mean vs persistence (%)'] = 100 * (1 - tab['Mean (km)'] / e_per.mean())
    tab['Skill RMSE vs persistence (%)'] = 100 * (1 - tab['RMSE (km)'] / rmse(e_per))
    comp = []
    for x, y in COMPARISONS:
        if x in E and y in E:
            dm, ci_m, dr, ci_r = boot_delta(E[x], E[y], groups)
            comp.append({'So sánh': f'{x} so với {y}', 'Δ RMSE (km)': dr, 'CI95 RMSE': f'[{ci_r[0]:.2f}, {ci_r[1]:.2f}]',
                         'RMSE: CI loại trừ 0': bool(ci_r[0] > 0 or ci_r[1] < 0),
                         'Δ Mean (km)': dm, 'CI95 Mean': f'[{ci_m[0]:.2f}, {ci_m[1]:.2f}]',
                         'Mean: CI loại trừ 0': bool(ci_m[0] > 0 or ci_m[1] < 0)})
    comp = pd.DataFrame(comp)
    full_tabs[fs] = (tab, comp)
    tab.round(4).to_csv(os.path.join(OUT_DIR, f'Bang_day_du_{6 * fs}h.csv'), index=False, encoding='utf-8-sig')
    comp.round(4).to_csv(os.path.join(OUT_DIR, f'So_sanh_cap_{6 * fs}h.csv'), index=False, encoding='utf-8-sig')

compact_df = pd.DataFrame(compact).T.set_index('Configuration')
compact_df.to_csv(os.path.join(OUT_DIR, 'Bang_ablation_RMSE.csv'), encoding='utf-8-sig')
print('=== Bảng ablation (RMSE, km, test; trung bình theo seed ± độ lệch chuẩn) ===')
display(compact_df)
for fs, (tab, comp) in full_tabs.items():
    print(f'\n========== +{6 * fs}h: bảng đầy đủ ==========')
    display(tab.set_index('Dòng').round(3))
    if len(comp):
        display(comp.set_index('So sánh').round(3))
print(f'Đã lưu CSV vào {OUT_DIR}/')

### 7.1 Biểu đồ RMSE theo dòng

In [ ]:
plot_tabs = {fs: v for fs, v in full_tabs.items() if 'Std RMSE' in v[0] and (v[0]['Configuration'] != 'Persistence').any()}
fig, axes = plt.subplots(1, max(len(plot_tabs), 1), figsize=(6.5 * max(len(plot_tabs), 1), 4.8), squeeze=False)
for ax, (fs, (tab, _)) in zip(axes[0], plot_tabs.items()):
    models = tab[tab['Configuration'] != 'Persistence'].reset_index(drop=True)
    pers = tab[tab['Configuration'] == 'Persistence'].iloc[0]
    bars = ax.bar(models['Dòng'], models['RMSE (km)'], yerr=models['Std RMSE'].fillna(0), width=0.6,
                  color='#4C78A8', edgecolor='white', capsize=4)
    for b, causal, v in zip(bars, models['Nhân quả'], models['RMSE (km)']):
        if not causal:
            b.set_hatch('//')
            b.set_alpha(0.55)
        ax.text(b.get_x() + b.get_width() / 2, v / 2, f'{v:.1f}', ha='center', va='center',
                color='white', fontsize=9, fontweight='bold')
    ax.axhline(pers['RMSE (km)'], color='#555555', ls='--', lw=1.5, label=f'Persistence {pers["RMSE (km)"]:.1f} km')
    ax.set_title(f'RMSE test +{6 * fs}h (sọc: không nhân quả)')
    ax.set_ylabel('km')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    ax.set_axisbelow(True)
    ax.legend(frameon=False, loc='upper right')
fig.tight_layout()
plt.show()